In [3]:
import os

import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

from tqdm import tqdm
from PIL import Image
from glob import glob
import importlib

import flow_matching_unet_model as flow_matching
importlib.reload(flow_matching)
ContextUnet = flow_matching.ContextUnet

In [5]:
class Config():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_feat = 128
    n_cfeat = 5
    height = 16

    timesteps = 500

    batch_size = 100
    n_epoch = 300
    lr = 1e-3

    image_size=64
    channels=3

    save_dir_weight = "flow_matching_weight"
    save_dir_img = "flow_matching_img"
    data_path = "../ddpm/datas/tinyhero"

    os.makedirs(save_dir_weight, exist_ok=True)
    os.makedirs(save_dir_img, exist_ok=True)

config = Config()

In [ ]:
def q_sample(x, t, noise):
    t = t[:, None, None, None]
    return t * x + (1 - t) * noise

@torch.no_grad()
def sample(model, imgs=None, n=16):
    model.eval()
    if imgs is None:
        imgs = torch.randn(n, config.channels, config.image_size, config.image_size, device=config.device)
    else:
        n = len(imgs)

    dt = 1.0 / config.timesteps

    for i in range(config.timesteps):
        t_input = torch.full((n, 1), i * dt, device=config.device, dtype=torch.float)
        v_preds = model(imgs, t_input)

        imgs += v_preds * dt

    imgs = (imgs.clamp(-1, 1) + 1) / 2
    return imgs

In [11]:
class HeroDataset(Dataset):
    def __init__(self, root, transform=None):
        self.paths = sorted(glob(os.path.join(root, "*/*.png")))
        self.transform= transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = HeroDataset(config.data_path, transform)
loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                    num_workers=4, pin_memory=True)

In [ ]:
def train():
    model = ContextUnet(config.channels, config.n_feat, config.n_cfeat, config.height)
    model.to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    schedule = torch.optim.lr_scheduler.StepLR(optimizer, step_size=60, gamma=0.1)

    for epoch in range(1, config.n_epoch + 1):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch} / {config.n_epoch}")

        for x in pbar:
            x = x.to(config.device)
            bs = x.size(0)

            t = torch.rand(bs, device=config.device)
            noise = torch.randn_like(x)

            x_t = q_sample(x, t, noise)
            t_input = t.unsqueeze(-1)
            vector_pred = model(x_t, t_input)

            loss = F.mse_loss(vector_pred, x - noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix({"Loss": loss.item()})

        schedule.step()

        imgs = sample(model, n=16)
        utils.save_image(imgs, f"{config.save_dir_img}/flow_matching_epoch_{epoch:03d}.png", nrow=4)
        if epoch % 30 == 0:
            torch.save(model.state_dict(), f"{config.save_dir_weight}/flow_matching_{epoch}epoch.pth")
    
    torch.save(model.state_dict(), f"{config.save_dir_weight}/flow_matching.pth")
    print("Training Complete")

In [13]:
train()

Epoch 299 / 300: 100%|██████████| 37/37 [00:11<00:00,  3.31it/s, Loss=0.0207]


Training Complete


In [18]:
len(loader), len(dataset)

(37, 3648)